# Data Collection & Preprocessing
## Air Quality Prediction in Houston, TX (COSC 4368 Final Project)
**Team 7**: Sadhana Bastola, Abdelrahman Darmousa, Tien Hoang, Tabriz Sadredinov

This notebook handles:
1. Downloading EPA air quality data for Harris County (Houston)
2. Pulling historical weather data from Open-Meteo
3. Merging everything together
4. Basic cleaning and saving a combined dataset


## Step 0: The Data

### EPA Air Quality Data (Manual Download)

It's in: https://aqs.epa.gov/aqsweb/airdata/download_files.html

Download the **daily** data files for 2022, 2023, 2024, and 2025, we need these specific ones:

| File Pattern | What It Is |
|---|---|
| `daily_aqi_by_county_YYYY.zip` | Daily AQI by county (our main target) |
| `daily_88101_YYYY.zip` | PM2.5 (FRM/FEM) |
| `daily_44201_YYYY.zip` | Ozone |
| `daily_42401_YYYY.zip` | SO2 |
| `daily_42101_YYYY.zip` | CO |
| `daily_42602_YYYY.zip` | NO2 |
| `daily_PRESS_YYYY.zip` | Barometric Pressure (if available) |
| `daily_TEMP_YYYY.zip` | Temperature from EPA stations |
| `daily_WIND_YYYY.zip` | Wind data |
| `daily_RH_DP_YYYY.zip` | Relative Humidity & Dew Point |



### Weather Data (Open-Meteo API)
No API key needed for this. The code below handles it.

###
- Harris County FIPS code is 48201 (State: 48 = Texas, County: 201 = Harris)
- We filter all EPA data for Harris County

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import requests
import time
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

## Step 1: Load and Filter EPA AQI Data
The main target variable is the daily AQI. We filter for Harris County, TX.

In [3]:
# Load all AQI by county files
aqi_files = sorted(glob.glob('data/raw/daily_aqi_by_county_*.csv'))
print(f"Found {len(aqi_files)} AQI files: {[os.path.basename(f) for f in aqi_files]}")

if len(aqi_files) == 0:
    print("No AQI files found")
else:
    aqi_dfs = []
    for f in aqi_files:
        df = pd.read_csv(f)
        # filter for Harris County, Texas
        df_harris = df[(df['State Name'] == 'Texas') & (df['county Name'] == 'Harris')].copy()
        aqi_dfs.append(df_harris)
        print(f"  {os.path.basename(f)}: {len(df_harris)} rows for Harris County")
    
    aqi_df = pd.concat(aqi_dfs, ignore_index=True)
    aqi_df['Date'] = pd.to_datetime(aqi_df['Date'])
    aqi_df = aqi_df.sort_values('Date').reset_index(drop=True)
    
    print(f"\nTotal AQI records: {len(aqi_df)}")
    print(f"Date range: {aqi_df['Date'].min()} to {aqi_df['Date'].max()}")
    print(f"\nColumns: {list(aqi_df.columns)}")
    aqi_df.head()

Found 4 AQI files: ['daily_aqi_by_county_2022.csv', 'daily_aqi_by_county_2023.csv', 'daily_aqi_by_county_2024.csv', 'daily_aqi_by_county_2025.csv']
  daily_aqi_by_county_2022.csv: 365 rows for Harris County
  daily_aqi_by_county_2023.csv: 365 rows for Harris County
  daily_aqi_by_county_2024.csv: 366 rows for Harris County
  daily_aqi_by_county_2025.csv: 274 rows for Harris County

Total AQI records: 1370
Date range: 2022-01-01 00:00:00 to 2025-10-01 00:00:00

Columns: ['State Name', 'county Name', 'State Code', 'County Code', 'Date', 'AQI', 'Category', 'Defining Parameter', 'Defining Site', 'Number of Sites Reporting']


## Step 2: Load Pollutant-Specific Data
We're going to extract daily averages per pollutant for Harris County.

In [8]:
def load_pollutant_data(file_pattern, pollutant_name):
    """Load a pollutant's daily data files and filter for Harris County."""
    files = sorted(glob.glob(f'data/raw/{file_pattern}'))
    if not files:
        print(f"  No files found for pattern {file_pattern}")
        return None
    
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        # filter for Harris County TX
        df_harris = df[(df['State Name'] == 'Texas') & (df['County Name'] == 'Harris')].copy()
        dfs.append(df_harris)
    
    if not dfs:
        return None
    
    combined = pd.concat(dfs, ignore_index=True)
    combined['Date Local'] = pd.to_datetime(combined['Date Local'], format='mixed')
    
    # average across all monitoring stations for each day
    # (Harris County has multiple stations)
    daily_avg = combined.groupby('Date Local').agg({
        'Arithmetic Mean': 'mean',
        '1st Max Value': 'max'
    }).reset_index()
    
    daily_avg.columns = ['Date', f'{pollutant_name}_mean', f'{pollutant_name}_max']
    print(f"  {pollutant_name}: {len(daily_avg)} daily records from {len(files)} files")
    return daily_avg

# load each pollutant
print("Loading pollutant data...")
pm25 = load_pollutant_data('daily_88101_*.csv', 'PM25')
ozone = load_pollutant_data('daily_44201_*.csv', 'Ozone')
so2 = load_pollutant_data('daily_42401_*.csv', 'SO2')
co = load_pollutant_data('daily_42101_*.csv', 'CO')
no2 = load_pollutant_data('daily_42602_*.csv', 'NO2')

# also load the meteorological ones from EPA
temp = load_pollutant_data('daily_TEMP_*.csv', 'Temp')
wind = load_pollutant_data('daily_WIND_*.csv', 'Wind')
rh = load_pollutant_data('daily_RH_DP_*.csv', 'RH')

Loading pollutant data...
  PM25: 1369 daily records from 4 files
  Ozone: 1369 daily records from 4 files
  SO2: 1368 daily records from 4 files
  CO: 1370 daily records from 4 files
  NO2: 1369 daily records from 4 files
  Temp: 1369 daily records from 4 files
  Wind: 1369 daily records from 4 files
  RH: 1369 daily records from 4 files


## Step 3: Merge Pollutant Data Together

Merge everything on the Date column. Some days are missing for certain pollutants

In [9]:
# start with AQI as the base
# keep the important columns from AQI
aqi_clean = aqi_df[['Date', 'AQI', 'Category', 'Defining Parameter', 'Number of Sites Reporting']].copy()

# some days might have multiple entries (if the county spans multiple reporting areas)
# take the max AQI for each day since that's what matters for health
aqi_daily = aqi_clean.groupby('Date').agg({
    'AQI': 'max',
    'Category': 'first',  # take whatever category corresponds
    'Number of Sites Reporting': 'max'
}).reset_index()

print(f"Unique days with AQI data: {len(aqi_daily)}")

# merge in each pollutant
merged = aqi_daily.copy()

pollutant_dfs = [
    ('PM25', pm25), ('Ozone', ozone), ('SO2', so2), 
    ('CO', co), ('NO2', no2),
    ('Temp', temp), ('Wind', wind), ('RH', rh)
]

for name, df in pollutant_dfs:
    if df is not None:
        merged = merged.merge(df, on='Date', how='left')
        print(f"Merged {name}: {df.shape[1]-1} new columns")
    else:
        print(f"Skipped {name} (no data)")

print(f"\nMerged shape: {merged.shape}")
print(f"Columns: {list(merged.columns)}")
merged.head()

Unique days with AQI data: 1370
Merged PM25: 2 new columns
Merged Ozone: 2 new columns
Merged SO2: 2 new columns
Merged CO: 2 new columns
Merged NO2: 2 new columns
Merged Temp: 2 new columns
Merged Wind: 2 new columns
Merged RH: 2 new columns

Merged shape: (1370, 20)
Columns: ['Date', 'AQI', 'Category', 'Number of Sites Reporting', 'PM25_mean', 'PM25_max', 'Ozone_mean', 'Ozone_max', 'SO2_mean', 'SO2_max', 'CO_mean', 'CO_max', 'NO2_mean', 'NO2_max', 'Temp_mean', 'Temp_max', 'Wind_mean', 'Wind_max', 'RH_mean', 'RH_max']


,Date,AQI,Category,Number of Sites Reporting,PM25_mean,PM25_max,Ozone_mean,Ozone_max,SO2_mean,SO2_max,CO_mean,CO_max,NO2_mean,NO2_max,Temp_mean,Temp_max,Wind_mean,Wind_max,RH_mean,RH_max
0,2022-01-01,64,Moderate,19,15.321094,50.1,0.019559,0.031,0.195259,6.1,0.287271,0.800,3.705903,23.1,76.576248,84.0,106.445473,359.8,72.684028,94.0
1,2022-01-02,38,Good,20,4.174053,26.2,0.027297,0.034,0.166786,0.8,0.160726,0.700,4.168877,14.8,40.471115,72.0,167.886952,356.7,41.546528,86.0
2,2022-01-03,51,Moderate,20,7.471324,30.0,0.014856,0.040,0.613545,2.8,0.374274,1.671,16.619978,42.6,41.105263,53.0,102.732751,359.9,35.041667,89.0
3,2022-01-04,65,Moderate,20,11.196814,33.0,0.026825,0.044,0.667277,4.6,0.658620,1.400,19.143158,52.1,52.067913,66.0,80.075478,358.9,52.403326,92.0
4,2022-01-05,77,Moderate,20,7.710880,52.0,0.024761,0.045,0.681042,10.7,0.447540,1.300,13.068277,40.2,63.983453,78.0,92.537610,335.3,66.832639,100.0


## Step 4: Pull Weather Data from Open-Meteo

Grab daily weather for Houston (lat 29.76, lon -95.37).

In [10]:
def fetch_open_meteo_weather(start_date, end_date, lat=29.7604, lon=-95.3698):

    # Open-Meteo has a limit on date range per request, so we chunk by year
    all_data = []
    
    current_start = pd.Timestamp(start_date)
    final_end = pd.Timestamp(end_date)
    
    while current_start < final_end:
        current_end = min(current_start + pd.DateOffset(months=6), final_end)
        
        params = {
            'latitude': lat,
            'longitude': lon,
            'start_date': current_start.strftime('%Y-%m-%d'),
            'end_date': current_end.strftime('%Y-%m-%d'),
            'daily': ','.join([
                'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
                'apparent_temperature_max', 'apparent_temperature_min',
                'precipitation_sum', 'rain_sum',
                'windspeed_10m_max', 'windgusts_10m_max', 'winddirection_10m_dominant',
                'shortwave_radiation_sum',
                'relative_humidity_2m_max', 'relative_humidity_2m_min', 'relative_humidity_2m_mean',
                'pressure_msl_mean',
                'cloud_cover_mean',
            ]),
            'timezone': 'America/Chicago',
            'temperature_unit': 'fahrenheit',
            'windspeed_unit': 'mph',
            'precipitation_unit': 'inch'
        }
        
        url = 'https://archive-api.open-meteo.com/v1/archive'
        print(f"  Fetching weather: {current_start.strftime('%Y-%m-%d')} to {current_end.strftime('%Y-%m-%d')}...")
        
        resp = requests.get(url, params=params)
        if resp.status_code == 200:
            data = resp.json()
            daily = data.get('daily', {})
            if daily and 'time' in daily:
                df_chunk = pd.DataFrame(daily)
                all_data.append(df_chunk)
            else:
                print(f"No daily data returned for this chunk")
        else:
            print(f"Error {resp.status_code}: {resp.text[:200]}")
        
        current_start = current_end + pd.DateOffset(days=1)
        time.sleep(0.5)  # be polite to the API
    
    if all_data:
        weather_df = pd.concat(all_data, ignore_index=True)
        weather_df['time'] = pd.to_datetime(weather_df['time'])
        weather_df = weather_df.rename(columns={'time': 'Date'})
        weather_df = weather_df.drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)
        return weather_df
    else:
        print("No weather data retrieved!")
        return None

# figure out date range from our AQI data
date_min = merged['Date'].min().strftime('%Y-%m-%d')
date_max = merged['Date'].max().strftime('%Y-%m-%d')
print(f"Fetching weather for {date_min} to {date_max}")

weather = fetch_open_meteo_weather(date_min, date_max)

if weather is not None:
    print(f"\nWeather data shape: {weather.shape}")
    print(f"Columns: {list(weather.columns)}")
    weather.head()

Fetching weather for 2022-01-01 to 2025-10-01
  Fetching weather: 2022-01-01 to 2022-07-01...
  Fetching weather: 2022-07-02 to 2023-01-02...
  Fetching weather: 2023-01-03 to 2023-07-03...
  Fetching weather: 2023-07-04 to 2024-01-04...
  Fetching weather: 2024-01-05 to 2024-07-05...
  Fetching weather: 2024-07-06 to 2025-01-06...
  Fetching weather: 2025-01-07 to 2025-07-07...
  Fetching weather: 2025-07-08 to 2025-10-01...

Weather data shape: (1370, 17)
Columns: ['Date', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'precipitation_sum', 'rain_sum', 'windspeed_10m_max', 'windgusts_10m_max', 'winddirection_10m_dominant', 'shortwave_radiation_sum', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'relative_humidity_2m_mean', 'pressure_msl_mean', 'cloud_cover_mean']


## Step 5: Merge Weather into the Main Dataset

In [12]:
if weather is not None:
    # merge weather data
    full_data = merged.merge(weather, on='Date', how='left')
    print(f"Final merged shape: {full_data.shape}")
else:
    print("No weather data lol")
    full_data = merged.copy()

# quick check on what we've got
print(f"\nDate range: {full_data['Date'].min()} to {full_data['Date'].max()}")
print(f"Total days: {len(full_data)}")
print(f"\nMissing values per column XD:")
print(full_data.isnull().sum().to_string())

Final merged shape: (1370, 36)

Date range: 2022-01-01 00:00:00 to 2025-10-01 00:00:00
Total days: 1370

Missing values per column XD:
Date                          0
AQI                           0
Category                      0
Number of Sites Reporting     0
PM25_mean                     1
PM25_max                      1
Ozone_mean                    1
Ozone_max                     1
SO2_mean                      2
SO2_max                       2
CO_mean                       0
CO_max                        0
NO2_mean                      1
NO2_max                       1
Temp_mean                     1
Temp_max                      1
Wind_mean                     1
Wind_max                      1
RH_mean                       1
RH_max                        1
temperature_2m_max            0
temperature_2m_min            0
temperature_2m_mean           0
apparent_temperature_max      0
apparent_temperature_min      0
precipitation_sum             0
rain_sum                      0
w

## Step 6: Cleaning Time!

Handle missing values, drop any columns that are mostly empty, etc.

In [15]:
# drop columns that are more than 50% missing
threshold = 0.5
cols_before = full_data.shape[1]
full_data = full_data.dropna(axis=1, thresh=int(len(full_data) * (1 - threshold)))
cols_after = full_data.shape[1]
print(f"Dropped {cols_before - cols_after} columns that were >50% missing")

# for the remaining missing values, we'll use forward fill then backward fill
# makes sense for time series, say if a sensor was down for a day, the previous day's reading is a decent estimate
numeric_cols = full_data.select_dtypes(include=[np.number]).columns
full_data[numeric_cols] = full_data[numeric_cols].ffill().bfill()

# check if there's still anything missing
remaining_missing = full_data.isnull().sum().sum()
print(f"Remaining missing values: {remaining_missing}")

if remaining_missing > 0:
    full_data = full_data.dropna()
    print(f"Dropped rows with remaining NaN. Final shape: {full_data.shape}")

full_data.head()

Dropped 0 columns that were >50% missing
Remaining missing values: 0


,Date,AQI,Category,Number of Sites Reporting,PM25_mean,PM25_max,Ozone_mean,Ozone_max,SO2_mean,SO2_max,...,rain_sum,windspeed_10m_max,windgusts_10m_max,winddirection_10m_dominant,shortwave_radiation_sum,relative_humidity_2m_max,relative_humidity_2m_min,relative_humidity_2m_mean,pressure_msl_mean,cloud_cover_mean
0,2022-01-01,64,Moderate,19,15.321094,50.1,0.019559,0.031,0.195259,6.1,...,0.063,21.6,35.1,203,10.81,96,55,82,1005.8,92
1,2022-01-02,38,Good,20,4.174053,26.2,0.027297,0.034,0.166786,0.8,...,0.028,22.2,36.9,331,13.69,97,40,56,1022.7,38
2,2022-01-03,51,Moderate,20,7.471324,30.0,0.014856,0.040,0.613545,2.8,...,0.000,9.6,16.6,345,14.75,68,37,52,1031.7,0
3,2022-01-04,65,Moderate,20,11.196814,33.0,0.026825,0.044,0.667277,4.6,...,0.000,11.8,21.0,185,13.94,82,46,64,1022.7,34
4,2022-01-05,77,Moderate,20,7.710880,52.0,0.024761,0.045,0.681042,10.7,...,0.000,13.2,21.9,203,13.45,99,53,83,1013.7,12


## Step 7: Add Time-Based Features

Day of week, month, season, etc. These capture patterns like weekday vs weekend traffic.

In [16]:
# time features
full_data['day_of_week'] = full_data['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
full_data['month'] = full_data['Date'].dt.month
full_data['day_of_year'] = full_data['Date'].dt.dayofyear
full_data['is_weekend'] = (full_data['day_of_week'] >= 5).astype(int)

# season (Houston doesn't really have 4 seasons but this still captures patterns)
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # winter
    elif month in [3, 4, 5]:
        return 1  # spring
    elif month in [6, 7, 8]:
        return 2  # summer
    else:
        return 3  # fall

full_data['season'] = full_data['month'].apply(get_season)

# is it a US federal holiday? (rough approximation)
# we'll just mark major ones manually, this not perfect but good enough
us_holidays_md = [
    (1, 1), (1, 15), (2, 19), (5, 27), (6, 19), (7, 4),
    (9, 2), (10, 14), (11, 11), (11, 28), (12, 25)
]
full_data['is_holiday'] = full_data.apply(
    lambda row: int((row['Date'].month, row['Date'].day) in us_holidays_md), axis=1
)

print("Added time features")
print(f"Final dataset shape: {full_data.shape}")
print(f"\nColumns: {list(full_data.columns)}")

Added time features
Final dataset shape: (1370, 42)

Columns: ['Date', 'AQI', 'Category', 'Number of Sites Reporting', 'PM25_mean', 'PM25_max', 'Ozone_mean', 'Ozone_max', 'SO2_mean', 'SO2_max', 'CO_mean', 'CO_max', 'NO2_mean', 'NO2_max', 'Temp_mean', 'Temp_max', 'Wind_mean', 'Wind_max', 'RH_mean', 'RH_max', 'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'precipitation_sum', 'rain_sum', 'windspeed_10m_max', 'windgusts_10m_max', 'winddirection_10m_dominant', 'shortwave_radiation_sum', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'relative_humidity_2m_mean', 'pressure_msl_mean', 'cloud_cover_mean', 'day_of_week', 'month', 'day_of_year', 'is_weekend', 'season', 'is_holiday']


## Step 8: Save the Processed Data

In [17]:
# save to csv
output_path = 'data/processed/houston_aqi_weather_combined.csv'
full_data.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {full_data.shape}")
print(f"\nQuick stats on AQI (target):")
print(full_data['AQI'].describe())

Saved to data/processed/houston_aqi_weather_combined.csv
Shape: (1370, 42)

Quick stats on AQI (target):
count    1370.000000
mean       69.669343
std        27.424286
min         5.000000
25%        54.000000
50%        62.000000
75%        76.000000
max       205.000000
Name: AQI, dtype: float64


download weather data manually (if something goes wrong):
- https://open-meteo.com/en/docs/historical-weather-api
- NOAA's Climate Data Online: https://www.ncdc.noaa.gov/cdo-web/
